# Circuit quantization of coupled Xoniums

Julia port of `xonium-readout/notebooks/xonium-couplers/xonium-taichi-couplers.py` (schematics omitted). Helpers live in `src/circuit_quantization.jl`.

A circuit is a `CGraph`: each `CNode` carries its capacitance to ground and each `CEdge` a coupling capacitor, all in fF. Charging energies come out in GHz and couplings in MHz.

In [ ]:
using LinearAlgebra
using Symbolics
using XoniumUnitCell

## Direct multi-path coupling

Nodes `A1`, `A2` belong to fluxonium A and `B1`, `B2` to fluxonium B. We change to sum and difference modes of each pair. The difference modes encode the qubits. The sum modes have no potential energy.

In [ ]:
pairs_direct = [:A => (:A1, :A2), :B => (:B1, :B2)]

### Capacitive coupling, symbolic

In [ ]:
@variables C1g C2g C3g C4g C_A C_c C_B
g_direct_sym = CGraph(
    [CNode(:A1, C1g), CNode(:A2, C2g), CNode(:B1, C3g), CNode(:B2, C4g)],
    [CEdge(:A1, :A2; C=C_A), CEdge(:A2, :B2; C=C_c), CEdge(:B1, :B2; C=C_B)],
)
modes_direct = sum_diff_transform(g_direct_sym, pairs_direct)
K_direct = transform_cap_matrix(cap_matrix(g_direct_sym), modes_direct)

### Numerics (fF)

In [ ]:
g_direct = CGraph(
    [CNode(:A1, 20.0), CNode(:A2, 80.0), CNode(:B1, 20.0), CNode(:B2, 80.0)],
    [CEdge(:A1, :A2; C=10.0), CEdge(:A2, :B2; C=5.0), CEdge(:B1, :B2; C=10.0)],
)
K_direct_num = transform_cap_matrix(cap_matrix(g_direct), modes_direct)
Cinv_params(inv(K_direct_num), modes_direct, :A, :B)

Dropping the sum modes *before* inverting changes the result: they load the qubit modes and change their effective mass. This underestimates $E_C$ and overestimates $J$.

In [ ]:
K_trunc = truncate_to_diff_modes(K_direct_num, modes_direct)
Cinv_params(inv(K_trunc), modes_direct, :A, :B; truncated=true)

## Direct multi-path coupling with diagonal capacitances

Add capacitors `A2`–`B1`, `A1`–`B2` and `A1`–`B1`. We want the effective coupling between the two difference modes.

In [ ]:
@variables C_A2B1 C_A1B2 C_A1B1
g_full_sym = CGraph(
    g_direct_sym.nodes,
    [g_direct_sym.edges; CEdge(:A2, :B1; C=C_A2B1); CEdge(:A1, :B2; C=C_A1B2); CEdge(:A1, :B1; C=C_A1B1)],
)
K_full = transform_cap_matrix(cap_matrix(g_full_sym), modes_direct)
Kinv_full = inv(K_full)
Symbolics.simplify.(Kinv_full[mode_index(modes_direct, :A), mode_index(modes_direct, :B)])

## Lumped resonator coupling (LRC)

Fluxonium A, a resonator `r` and fluxonium B form a chain of three node pairs. The layout is mirror-symmetric, so `B1` has the ground capacitance of `A2` and `B2` that of `A1`. Sharing those variables replaces the substitution step of the original notebook.

In [ ]:
@variables C1g C2g C3g C4g C_A C_c1 C_r C_c2 C_B
g_lrc_sym = CGraph(
    [CNode(:A1, C1g), CNode(:A2, C2g), CNode(:r1, C3g), CNode(:r2, C4g),
     CNode(:B1, C2g), CNode(:B2, C1g)],
    [CEdge(:A1, :A2; C=C_A), CEdge(:A2, :r2; C=C_c1), CEdge(:r1, :r2; C=C_r),
     CEdge(:r2, :B2; C=C_c2), CEdge(:B1, :B2; C=C_B)],
)
pairs_lrc = [:A => (:A1, :A2), :r => (:r1, :r2), :B => (:B1, :B2)]
modes_lrc = sum_diff_transform(g_lrc_sym, pairs_lrc)
K_lrc = transform_cap_matrix(cap_matrix(g_lrc_sym), modes_lrc)

In [ ]:
Kinv_lrc = inv(K_lrc)
Kinv_lrc[mode_index(modes_lrc, :A), mode_index(modes_lrc, :B)]   # A–B coupling through the resonator

Numerics for the LRC layout. The resonator ground capacitance comes from `CPW_freq2cap`. Fill in the capacitances below.

In [ ]:
C_res = CPW_freq2cap(7.0)   # 7 GHz λ/4 resonator
g_lrc = CGraph(
    [CNode(:A1, 20.0), CNode(:A2, 80.0), CNode(:r1, C_res / 2), CNode(:r2, C_res / 2),
     CNode(:B1, 80.0), CNode(:B2, 20.0)],
    [CEdge(:A1, :A2; C=10.0), CEdge(:A2, :r2; C=5.0), CEdge(:r1, :r2; C=10.0),
     CEdge(:r2, :B2; C=5.0), CEdge(:B1, :B2; C=10.0)],
)
Kinv_lrc_num = inv(transform_cap_matrix(cap_matrix(g_lrc), modes_lrc))
(
    qubits = Cinv_params(Kinv_lrc_num, modes_lrc, :A, :B),
    qubit_resonator_g = Cinv2g(Kinv_lrc_num[mode_index(modes_lrc, :A), mode_index(modes_lrc, :r)]),
)